# HW4 (c): Text-to-Speech with Language Models (40 points)

## Introduction

In this assignment, you will implement and train a Language Model (LM) for Text-to-Speech (TTS) synthesis, following the CosyVoice2 architecture. The assignment focuses on the core component of modern TTS systems: autoregressive language modeling for speech token generation.

### Learning Objectives

* Understand modern TTS architecture and how LLM-based TTS systems work
* Implement autoregressive generation with a transformer that generates speech tokens from text
* Handle multi-modal sequences by working with text and speech tokens in a unified framework
* Train a language model from scratch on real speech data
* Implement zero-shot voice cloning with in-context learning
* Evaluate TTS quality using Word Error Rate (WER) with automatic speech recognition

### System Architecture

```
Text → [Text Tokenizer] → Text Tokens → [Your LM] → Speech Tokens → [Flow Model] → Mel → [Vocoder] → Audio
```

You will implement and train the Language Model component that converts text tokens to speech tokens. All other components (tokenizers, flow model, vocoder) are provided as pre-trained models.

### Assignment Package

Download [**hw4_util.zip**](https://drive.google.com/file/d/1q5OdFHgXBtdK5MHJCWmWeyBHmz4Y-QVy/view?usp=sharing) and upload it to your Google Drive.

### Dataset

The assignment uses the LibriTTS dataset (pre-tokenized for efficiency):
* 354,780 training samples
* 9,957 test samples
* Multi-speaker data for diverse voice generation

### Resources

- [CosyVoice2 Paper](https://arxiv.org/abs/2412.10117)
- [LibriTTS Dataset](https://www.openslr.org/60/)
- [Transformer Architecture](https://arxiv.org/abs/1706.03762)

### Submission Requirements

You will submit **only 2 files** to Gradescope:

1. **`submission_[YOUR_ID].txt`** - Auto-generated WER evaluation results from Part 8
2. **`hw4-c.pdf`** - PDF export of this notebook showing all your code and outputs

**Important Notes:**
* The submission file is automatically generated when you run Part 8's evaluation
* Do NOT modify the submission file format - it must match the exact format for autograding
* Your WER score will be calculated on Gradescope using your transcriptions
* Ensure all code cells have been executed and outputs are visible in the PDF


## Part 0: Environment Setup

The cell below will automatically detect your environment (Google Colab or Local Machine) and set it up accordingly.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/hw4_util /content/__MACOSX
!unzip -o -q /content/drive/MyDrive/hw4_util.zip -d /content
# The archive nests everything under hw4_util/. Lifting the contents up to /content
# and deleting that directory is what stops `import hw4_util` from resolving to the
# directory (a namespace package) instead of hw4_util.py.
!cp -r /content/hw4_util/. /content/ && rm -rf /content/hw4_util /content/__MACOSX
!ls /content
!pip install -q -r /content/requirements.txt

# Environment Verification
import sys, importlib

if '/content' not in sys.path:
    sys.path.insert(0, '/content')
# A failed earlier import leaves a namespace package cached under the same name
sys.modules.pop('hw4_util', None)
importlib.invalidate_caches()

from hw4_util import check_environment

print(f"hw4_util loaded from: {sys.modules['hw4_util'].__file__}")
check_environment()

Mounted at /content/drive
CosyVoice  hw4_util.py		 requirements.txt
drive	   libritts_token_cache  sample_data
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 53.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 23.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

True

## Part 1: Setup and Configuration

In this part, you will set up the environment and define the configuration for the assignment.


In [2]:
# Essential imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import unpad_sequence, pad_sequence
import numpy as np
import math
from tqdm.notebook import tqdm
from typing import Tuple
from IPython.display import Audio, display
import torchaudio

# Set environment variables
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Check GPU availability
assert torch.cuda.is_available(), "GPU is required for this assignment!"
device = torch.device('cuda')
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [3]:
# Configuration
class Config:
    """Essential configuration for HW4 - Paths and Fixed Tokens Only"""

    # Paths (Fixed - Do Not Change)
    DATA_CACHE_DIR = './libritts_token_cache'
    TRAIN_CACHE = 'libritts_train.pt'
    TEST_CACHE = 'libritts_test.pt'
    PRETRAINED_DIR = './pretrained_models'
    RESULTS_DIR = './results'

    # Special tokens (Fixed - Matching CosyVoice2)
    IGNORE_ID = -1        # Padding/ignore token for loss
    SOS_EOS_ID = 0        # Start/End of sequence token
    TASK_ID = 1           # Task separator token

    # Device configuration
    DEVICE = 'cuda'

# Create necessary directories
import os
os.makedirs(Config.RESULTS_DIR, exist_ok=True)
os.makedirs(Config.PRETRAINED_DIR, exist_ok=True)

print(f"Configuration loaded")
print(f"Data directory: {Config.DATA_CACHE_DIR}")
print(f"Results directory: {Config.RESULTS_DIR}")


Configuration loaded
Data directory: ./libritts_token_cache
Results directory: ./results


## Part 2: Load Pretrained Components

In this section, you will load the pre-trained [CosyVoice2](https://arxiv.org/abs/2412.10117) components that remain frozen during training:

* **Text Tokenizer**: Qwen2 BPE tokenizer for converting text to tokens
* **Speech Tokenizer**: VQ-VAE for converting audio to discrete speech tokens  
* **Flow Matching Model**: For converting speech tokens to mel-spectrograms (inference only)
* **Vocoder**: HiFi-GAN for converting mel-spectrograms to audio waveforms (inference only)


In [4]:
# 1. Replace the CUDA-13 build with the CPU build
!pip uninstall -y -q onnxruntime-gpu onnxruntime
!pip install -q onnxruntime

# 2. Clear the failed import out of the live session
import sys, importlib
for _name in [m for m in sys.modules if m == 'onnxruntime' or m.startswith('onnxruntime.')]:
    del sys.modules[_name]
importlib.invalidate_caches()

# 3. Import and make provider selection non-fatal
import onnxruntime as ort

_original_session = ort.InferenceSession

def _session_with_available_providers(*args, **kwargs):
    available = set(ort.get_available_providers())
    requested = kwargs.get('providers')
    if requested:
        kept = [p for p in requested
                if (p[0] if isinstance(p, (list, tuple)) else p) in available]
        kwargs['providers'] = kept or ['CPUExecutionProvider']
    return _original_session(*args, **kwargs)

ort.InferenceSession = _session_with_available_providers

print("onnxruntime version:", ort.__version__)
print("available providers:", ort.get_available_providers())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 89.4 MB/s eta 0:00:00
onnxruntime version: 1.28.0
available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [5]:
# Import the pretrained model utilities
# Note: This module is provided and should NOT be modified
from hw4_util import (
    download_pretrained_models,
    load_text_tokenizer,
    load_speech_tokenizer,
    load_flow_model,
    load_vocoder
)

# Download pretrained models if needed
print("Downloading pretrained CosyVoice2 models...")
model_dir = download_pretrained_models(Config.PRETRAINED_DIR)
print(f"Models ready at: {model_dir}")

# Load tokenizers (needed for training)
print("\nLoading tokenizers for training...")
text_tokenizer = load_text_tokenizer(model_dir)
speech_tokenizer = load_speech_tokenizer(model_dir, device)

print(f"Text tokenizer loaded")
print(f"Vocab size: {text_tokenizer.vocab_size}")
print(f"Speech tokenizer loaded")
print(f"Vocab size: {speech_tokenizer.vocab_size}")

# Note: Flow model and vocoder will be loaded later for inference only


2026-07-26 16:21:43,172 | INFO    | modelscope_hub.download | Downloading 21 files from iic/CosyVoice2-0.5B@master


Downloading:   0%|          | 0/21 [00:00<?, ?file/s]

.gitattributes:   0%|          | 0.00/3.12k [00:00<?, ?B/s]

configuration.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

campplus.onnx:   0%|          | 0.00/28.3M [00:00<?, ?B/s]

...

speech_tokenizer_v2.batch.onnx:   0%|          | 0.00/496M [00:00<?, ?B/s]

speech_tokenizer_v2.onnx:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Downloaded to: pretrained_models/models/iic--CosyVoice2-0.5B/snapshots/master
Models ready at: pretrained_models/models/iic--CosyVoice2-0.5B/snapshots/master

Loading tokenizers for training...
Text tokenizer loaded (Qwen2 BPE)
  Vocab size: 151643
Text tokenizer loaded
Vocab size: 151643
Speech tokenizer loaded
Vocab size: 6561


## Part 3: Dataset and Data Loading

In this part, you will implement a custom dataset class for the LibriTTS data. The dataset uses pre-tokenized speech data for efficiency.

### Requirements:
* Implement text tokenization (on-the-fly)
* Load pre-computed speech tokens from cache
* Handle multi-speaker information
* Implement proper sequence padding and batching

**TODO:** Complete the `__getitem__` method in the CosyVoiceDataset class below.

**TODO:** Implement the `cosyvoice_collate_fn` function for batching.


In [19]:
class CosyVoiceDataset(Dataset):
    """
    Multi-speaker TTS Dataset for CosyVoice2 using pre-tokenized LibriTTS data

    This dataset uses pre-computed speech tokens from cache, avoiding
    on-the-fly audio processing for faster training.
    """
    def __init__(self, samples_list, text_tokenizer, speaker_to_idx_dict,
                 max_text_len=200, max_speech_len=500):
        """
        Args:
            samples_list: List of pre-processed samples from .pt file
            text_tokenizer: Pretrained text tokenizer
            speaker_to_idx_dict: Speaker ID to index mapping from metadata
            max_text_len: Maximum text token length
            max_speech_len: Maximum speech token length
        """
        self.samples = samples_list
        self.text_tokenizer = text_tokenizer
        self.speaker_to_idx = speaker_to_idx_dict
        self.max_text_len = max_text_len
        self.max_speech_len = max_speech_len
        self.num_speakers = len(speaker_to_idx_dict)

        print(f"  Dataset initialized with {len(self.samples)} samples")
        print(f"  Number of speakers: {self.num_speakers}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Get a single sample

        Returns:
            dict with keys: 'utt', 'text', 'text_token', 'speech_token',
                           'speaker_idx', 'speaker_id'
        """
        # TODO: Implement the __getitem__ method
        #
        # Tasks:
        # 1. Get sample from self.samples[idx]
        sample = self.samples[idx]
        # 2. Extract: text, speaker_id, speech_tokens (pre-computed), utt_id
        text = sample['text']
        speaker_id = sample['speaker_id']
        speech_tokens = sample['speech_tokens']
        utt_id = sample['utt_id']
        # 3. Return None if text or speech_tokens are invalid/empty
        if not text or not speech_tokens:
            return None
        text = str(text).strip()
        if len(text) == 0 :
            return None
        # 4. Get speaker index from self.speaker_to_idx dictionary
        speaker_idx = self.speaker_to_idx.get(speaker_id, 0)

        # 5. Tokenize text and truncate to max_text_len
        try:
            text_token = self.text_tokenizer.encode(text)
        except Exception:
            return None
        if hasattr(text_token, 'ids'):
            text_token = text_token.ids
        text_token = torch.as_tensor(text_token, dtype=torch.long).flatten()[:self.max_text_len]
        if text_token.numel() == 0:
            return None
        # 6. Convert speech_tokens to tensor and truncate to max_speech_len
        speech_token = torch.as_tensor(speech_tokens,
                                       dtype=torch.long).flatten()[:self.max_speech_len]

        # 7. Return dict with required keys

        return {
            'utt': utt_id,
            'text': text,
            'text_token': text_token,
            'speech_token': speech_token,
            'speaker_idx': speaker_idx,
            'speaker_id': speaker_id,
        }

In [20]:
# Load the cache only if it isn't already in memory (it's big; avoid loading twice)
if 'train_data' not in globals():
    print("Loading LibriTTS cache...")
    train_cache = torch.load(f'{Config.DATA_CACHE_DIR}/{Config.TRAIN_CACHE}')
    test_cache = torch.load(f'{Config.DATA_CACHE_DIR}/{Config.TEST_CACHE}')

    train_data = train_cache['samples']
    val_data = test_cache['samples']
    metadata = train_cache['metadata']
    speaker_to_idx = metadata['speaker_to_idx']
    speech_vocab_size = metadata['speech_vocab_size']
    print("done")

s = train_data[0]
print("KEYS: ", list(s.keys()))
print("TYPES:", {k: type(v).__name__ for k, v in s.items()})
print()
print("speech_vocab_size:", speech_vocab_size)
print("speaker_to_idx size:", len(speaker_to_idx))
print("train speaker in map?", s.get('speaker_id') in speaker_to_idx)

v = val_data[0]
print("val speaker in map?  ", v.get('speaker_id') in speaker_to_idx)

KEYS:  ['utt_id', 'split', 'text', 'speech_tokens', 'speaker_id']
TYPES: {'utt_id': 'str', 'split': 'str', 'text': 'str', 'speech_tokens': 'list', 'speaker_id': 'str'}

speech_vocab_size: 6561
speaker_to_idx size: 247
train speaker in map? True
val speaker in map?   False


In [21]:
def cosyvoice_collate_fn(batch):
    """
    Collate function for batching variable-length sequences.
    Adapted from CosyVoice's collate strategy.

    Key features:
    - Filters invalid samples (None)
    - Sorts by speech length (descending) to minimize padding
    - Pads sequences efficiently
    """
    # TODO: Implement the collate function
    #
    # Tasks:
    # 1. Filter out None samples
    batch = [sample for sample in batch if sample is not None]
    if len(batch) == 0:
        return None

    # 2. Sort samples by speech_token length (descending)
    batch = sorted(batch, key=lambda s: s['speech_token'].size(0), reverse=True)

    # 3. Extract and reorder: utts, text, speaker_indices
    utts = [s['utt'] for s in batch]
    text = [s['text'] for s in batch]
    speaker_indices = torch.tensor([s['speaker_idx'] for s in batch], dtype=torch.long)
    text_token_list = [s['text_token'] for s in batch]
    speech_token_list = [s['speech_token'] for s in batch]

    # 4. Pad text_tokens and speech_tokens to same length within batch
    text_tokens   = pad_sequence(text_token_list,   batch_first=True, padding_value=0)
    speech_tokens = pad_sequence(speech_token_list, batch_first=True, padding_value=0)

    # 5. Create length tensors for actual sequence lengths

    text_lengths = torch.tensor([t.size(0) for t in text_token_list], dtype=torch.int32)
    speech_lengths = torch.tensor([t.size(0) for t in speech_token_list], dtype=torch.int32)

    # Use padding_value=0 for token sequences
    # Return dict with keys: utts, text, text_tokens, text_lengths,
    #                        speech_tokens, speech_lengths, speaker_indices

    return {
        'utts': utts,
        'text': text,
        'text_tokens': text_tokens,
        'text_lengths': text_lengths,
        'speech_tokens': speech_tokens,
        'speech_lengths': speech_lengths,
        'speaker_indices': speaker_indices,
    }

In [22]:
# Re-create datasets from the updated class (cheap - no cache reload)
train_dataset = CosyVoiceDataset(train_data, text_tokenizer, speaker_to_idx)
val_dataset = CosyVoiceDataset(val_data, text_tokenizer, speaker_to_idx)

BATCH_SIZE = 32   # A100; drop to 16 if Part 6 hits OOM

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    collate_fn=cosyvoice_collate_fn,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    collate_fn=cosyvoice_collate_fn,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

print(f"Train batches: {len(train_loader)}")   # expect 11087
print(f"Val batches: {len(val_loader)}")       # expect 312

  Dataset initialized with 354780 samples
  Number of speakers: 247
  Dataset initialized with 9957 samples
  Number of speakers: 247
Train batches: 11087
Val batches: 312


In [23]:
b = next(iter(train_loader))
print({k: (tuple(v.shape) if torch.is_tensor(v) else len(v)) for k, v in b.items()})
print("speech_lengths:", b['speech_lengths'][:8].tolist())
print("text_lengths:  ", b['text_lengths'][:8].tolist())
print("sample text:   ", b['text'][0][:60])

{'utts': 32, 'text': 32, 'text_tokens': (32, 86), 'text_lengths': (32,), 'speech_tokens': (32, 500), 'speech_lengths': (32,), 'speaker_indices': (32,)}
speech_lengths: [500, 413, 277, 267, 266, 242, 232, 216]
text_lengths:   [86, 52, 42, 43, 45, 22, 25, 24]
sample text:    Cut these three wands up from below, and strike with them up


In [24]:
vb = next(iter(val_loader))
print("val ok:", tuple(vb['speech_tokens'].shape), "| speakers:", vb['speaker_indices'][:5].tolist())

val ok: (32, 500) | speakers: [0, 0, 0, 0, 0]


## Part 4: Model Architecture

In this part, you will implement the **TextToSpeechLM** model - the core component of the TTS system.

### Model Requirements:
1. Takes text tokens as input
2. Uses transformer layers to process the sequence
3. Generates speech tokens autoregressively
4. Follows the CosyVoice2 sequence format: `[SOS, text_tokens, TASK_ID, speech_tokens]`

**TODO:** Complete the model architecture with proper embeddings and transformer layers.

**TODO:** Implement the forward pass with attention masking and loss computation.

**TODO:** Implement the generate method for autoregressive inference.


In [25]:
class TextToSpeechLM(nn.Module):
    """Student implementation of text to speech token generation model

    Architecture:
    - Text tokens → Text embeddings → Transformer → Speech tokens
    - Special tokens: SOS_EOS (id=0), TASK_ID (id=1)
    - Sequence format: [SOS, text_tokens, TASK_ID, speech_tokens]
    """

    def __init__(self,
                 text_vocab_size: int,
                 speech_vocab_size: int,
                 d_model: int = 768,
                 n_heads: int = 12,
                 n_layers: int = 12,
                 max_seq_len: int = 2048):
        super().__init__()

        self.d_model = d_model
        self.speech_vocab_size = speech_vocab_size

        # Special token IDs
        self.sos_eos_id = Config.SOS_EOS_ID
        self.task_id = Config.TASK_ID

        self.max_seq_len = max_seq_len

        # TODO: Define the model architecture
        #
        # Components needed:
        # 1. Text embedding layer (vocab_size → d_model)
        self.text_vocab_size = text_vocab_size + 256
        self.text_embedding = nn.Embedding(self.text_vocab_size, d_model)

        # 2. Speech embedding layer (vocab_size+1 → d_model, +1 for EOS)
        self.speech_embedding = nn.Embedding(speech_vocab_size + 1, d_model)

        # 3. Special token embeddings (2 tokens)
        self.special_embedding = nn.Embedding(2, d_model)

        # 4. Positional encoding (learnable parameters)
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_seq_len, d_model))

        # 5. Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            norm=nn.LayerNorm(d_model),
        )
        self.dropout = nn.Dropout(0.1)

        # 6. Output projection (d_model → speech_vocab_size+1)

        self.output_proj = nn.Linear(d_model, speech_vocab_size + 1)

        def _init_weights(module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
        self.apply(_init_weights)
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

        # Loss function (provided)
        self.criterion = nn.CrossEntropyLoss(ignore_index=Config.IGNORE_ID)

    def prepare_sequence(self, text_tokens, text_lengths, speech_tokens=None, speech_lengths=None):
        """Prepare input sequence in CosyVoice2 format

        Args:
            text_tokens: Text token ids [B, T_text]
            text_lengths: Actual lengths of text [B]
            speech_tokens: Speech token ids [B, T_speech] (training only)
            speech_lengths: Actual lengths of speech [B] (training only)

        Returns:
            lm_input: Model input embeddings
            lm_target: Target token ids for loss computation
            padding_mask: Boolean mask for padded positions
        """
        # TODO: Implement sequence preparation
        device = text_tokens.device
        batch_size = text_tokens.size(0)
        is_training = speech_tokens is not None

        # Tasks:
        # 1. Get embeddings for text, speech (if training), and special tokens
        sos_emb = self.special_embedding.weight[self.sos_eos_id].unsqueeze(0)
        task_emb = self.special_embedding.weight[self.task_id].unsqueeze(0)

        # 4. Handle variable lengths using unpad/pad operations
        text_lengths = text_lengths.to(torch.int64)
        text_emb = unpad_sequence(self.text_embedding(text_tokens),
                                  text_lengths.cpu(), batch_first=True)
        if is_training:
            speech_lengths = speech_lengths.to(torch.int64)
            speech_emb = unpad_sequence(self.speech_embedding(speech_tokens),
                                        speech_lengths.cpu(), batch_first=True)
        sequences, targets = [], []
        for i in range(batch_size):
            n_text = int(text_lengths[i])

            # 2. Build input sequence: [SOS, text_emb, TASK, speech_emb]
            if not is_training:
                sequences.append(torch.cat([sos_emb, text_emb[i], task_emb], dim=0))
                continue
            n_speech = int(speech_lengths[i])
            sequences.append(
                torch.cat([sos_emb, text_emb[i], task_emb, speech_emb[i]], dim=0)
            )

            # 3. Build target sequence for teacher forcing
            targets.append(torch.cat([
                torch.full((1 + n_text,), Config.IGNORE_ID, dtype=torch.long, device=device),
                speech_tokens[i, :n_speech].long(),
                torch.tensor([self.speech_vocab_size], dtype=torch.long, device=device),
            ], dim=0))

        lengths = torch.tensor([s.size(0) for s in sequences], dtype=torch.long, device=device)
        lm_input = pad_sequence(sequences, batch_first=True, padding_value=0.0)


        # 5. Create padding mask (True where padded)
        padding_mask = torch.arange(lm_input.size(1), device=device).unsqueeze(0) >= lengths.unsqueeze(1)
        lm_target = None
        if is_training:
            lm_target = pad_sequence(targets, batch_first=True, padding_value=Config.IGNORE_ID)
        return lm_input, lm_target, padding_mask

        # Note: Target should be shifted for next-token prediction
        # Note: Use Config.IGNORE_ID for positions to ignore in loss

    def _decode(self, lm_input, padding_mask=None):
        """Shared by forward() and generate() so both use identical logic
        Covers forward-pass steps 2-5: positional encoding, causal mask,
        transformer, output projection.
        """
        seq_len = lm_input.size(1)
        # 2. Add positional encoding to embeddings
        hidden = self.dropout(lm_input + self.pos_embedding[:, :seq_len])

        # 3. Create causal mask for autoregressive modeling (True = cannot attend)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=lm_input.device),
            diagonal=1,
        )

        # 4. Pass through transformer with both masks
        hidden = self.transformer(hidden, mask=causal_mask, src_key_padding_mask=padding_mask)

        # 5. Project to output vocabulary
        return self.output_proj(hidden)

    def forward(self, text_tokens, text_lengths, speech_tokens, speech_lengths):
        """Forward pass for training

        Args:
            text_tokens: [B, T_text] padded text tokens
            text_lengths: [B] actual lengths
            speech_tokens: [B, T_speech] padded speech tokens
            speech_lengths: [B] actual lengths
        """
        # TODO: Implement forward pass
        #
        # Steps:
        # 1. Prepare sequences using prepare_sequence
        lm_input, lm_target, padding_mask = self.prepare_sequence(
            text_tokens, text_lengths, speech_tokens, speech_lengths
        )

        # 2. Add positional encoding to embeddings
        # 3. Create causal mask for autoregressive modeling
        # 4. Pass through transformer with both masks
        # 5. Project to output vocabulary
        logits = self._decode(lm_input, padding_mask)

        # 6. Compute loss using targets from prepare_sequence
        loss = self.criterion(logits.reshape(-1, logits.size(-1)).float(), lm_target.reshape(-1))

        # 7. Compute accuracy: (correct predictions) / (non-ignored positions)
        with torch.no_grad():
            scored = lm_target != Config.IGNORE_ID
            correct = (logits.argmax(dim=-1) == lm_target) & scored
            accuracy = correct.sum().float() / scored.sum().clamp(min=1)
        # Return: loss (scalar), accuracy (scalar)

        return loss, accuracy

    @staticmethod
    def _as_batch(tokens, device):
        if not torch.is_tensor(tokens):
            tokens = torch.tensor(tokens)
        return tokens.to(device=device, dtype=torch.long).reshape(1, -1)

    @staticmethod
    def _sample_token(logits, temperature, top_k, history, win_size=10, tau_r=0.1):
        probs = torch.softmax(logits / max(float(temperature), 1e-5), dim=-1)

        if top_k is not None and 0 < top_k < probs.numel():
            top_probs, top_ids = probs.topk(int(top_k))
            token = int(top_ids[torch.multinomial(top_probs, num_samples=1)])
        else:
            token = int(torch.multinomial(probs, num_samples=1))

        if len(history) >= win_size and history[-win_size:].count(token) >= win_size * tau_r:
            token = int(torch.multinomial(probs, num_samples=1))

        return token

    @torch.no_grad()
    def generate(self, text_tokens, max_length=500, temperature=1.0, top_k=50,
                 prompt_speech_token=None, prompt_text_tokens=None, min_length=None):
        """Generate speech tokens autoregressively

        Args:
            text_tokens: Input text [1, T]
            max_length: Maximum generation length
            temperature: Sampling temperature
            top_k: Top-k sampling
            prompt_speech_token: Optional voice prompt
            prompt_text_tokens: Optional text for voice prompt
        """
        # TODO: Implement autoregressive generation
        #
        # Steps:

        self.eval()
        device = next(self.parameters()).device
        eos_id = self.speech_vocab_size
        text_tokens = self._as_batch(text_tokens, device)
        n_target_text = text_tokens.size(1)

        # 2. Add voice prompt if provided (for voice cloning)

        if prompt_text_tokens is not None:
            text_tokens = torch.cat([self._as_batch(prompt_text_tokens, device), text_tokens], dim=1)

        # 1. Build initial sequence with special tokens: [SOS, text, TASK, (prompt speech)]
        sos_emb = self.special_embedding.weight[self.sos_eos_id].view(1, 1, -1)
        task_emb = self.special_embedding.weight[self.task_id].view(1, 1, -1)

        parts = [sos_emb, self.text_embedding(text_tokens), task_emb]
        if prompt_speech_token is not None:
            prompt_speech = self._as_batch(prompt_speech_token, device)
            if prompt_speech.numel() > 0:
                parts.append(self.speech_embedding(prompt_speech))
        lm_input = torch.cat(parts, dim=1)

        if min_length is None:
            min_length = min(int(max_length), 2 * n_target_text)

        use_amp = device.type == 'cuda'
        generated = []

        # 3. Generation loop
        for _ in range(int(max_length)):
            if lm_input.size(1) >= self.max_seq_len:
                break
            #    - Add positional encoding
            #    - Create causal mask
            #    - Forward through transformer
            #    - Get logits for last position
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = self._decode(lm_input)[0, -1]
            logits = logits.float()

            if len(generated) < min_length:
                logits[eos_id] = float('-inf')

            #    - Apply temperature and top-k sampling
            next_token = self._sample_token(logits, temperature, top_k, generated)

            #    - Stop at EOS token (speech_vocab_size)
            if next_token == eos_id:
                break

            generated.append(next_token)
            next_emb = self.speech_embedding(
                torch.tensor([[next_token]], dtype=torch.long, device=device)
            )
            lm_input = torch.cat([lm_input, next_emb], dim=1)

            # 4. Return generated token ids
            #
            # Note: Prevent EOS before min_length tokens

        return torch.tensor(generated, dtype=torch.long, device=device)


In [26]:
# Initialize model
print("Initializing TextToSpeechLM...")
model = TextToSpeechLM(
    text_vocab_size=text_tokenizer.vocab_size,
    speech_vocab_size=speech_vocab_size
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {num_params:,} ({num_params/1e6:.1f}M)")
print(f"Trainable: {trainable:,} ({trainable/1e6:.1f}M)")


Initializing TextToSpeechLM...


/tmp/ipykernel_1240/2415452517.py:54: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Total parameters: 213,374,626 (213.4M)
Trainable: 213,374,626 (213.4M)


## Part 5: Sanity Check - Verify Data Pipeline

Before training, verify that your data processing pipeline works correctly by generating audio from ground-truth speech tokens. This section helps you debug any issues before starting the training process.


In [37]:
# Audio generation helper (imported from hw4_util)
from hw4_util import generate_audio_from_tokens

# Run sanity check - generate audio from ground-truth tokens
print("Running sanity check...")

# Load flow model and vocoder for sanity check
print("Loading flow model and vocoder for sanity check...")
flow_model = load_flow_model(model_dir, device)
vocoder = load_vocoder(model_dir, device)

# Get a sample from the dataset
sample_idx = 0
sample = val_dataset[sample_idx]

if sample is not None:
    print(f"\n Sample text: '{sample['text']}'")
    print(f"  Text tokens: {sample['text_token'].shape}")
    print(f"  Speech tokens: {sample['speech_token'].shape}")

    # Generate audio from ground-truth speech tokens
    audio, mel = generate_audio_from_tokens(
        sample['speech_token'],
        flow_model,
        vocoder,
        device
    )

    sample_rate = 24000
    duration = audio.shape[0] / sample_rate

    print(f"\n Generated audio from ground-truth tokens:")
    print(f"  Mel shape: {mel.shape}")
    print(f"  Audio shape: {audio.shape}")
    print(f"  Duration: {duration:.2f}s @ {sample_rate}Hz")

    # Save for listening
    output_path = f'{Config.RESULTS_DIR}/sanity_check.wav'
    torchaudio.save(output_path, audio.cpu().unsqueeze(0), sample_rate)
    print(f"  Saved to: {output_path}")

    # Display audio in notebook
    display(Audio(audio.cpu().numpy(), rate=sample_rate))

    print("\n Sanity check passed! Data pipeline is working correctly.")
else:
    print("Sample is None - check dataset configuration")

# Clean up to save memory
del flow_model
del vocoder
torch.cuda.empty_cache()


Running sanity check...
Loading flow model and vocoder for sanity check...


/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


  - Checkpoint: flow.pt
  - Device: cuda
  - Checkpoint: hift.pt
  - Device: cuda

 Sample text: 'He defended Raglan Castle to extremity; and opened not its gates till the middle of August.'
  Text tokens: torch.Size([20])
  Speech tokens: torch.Size([176])

 Generated audio from ground-truth tokens:
  Mel shape: torch.Size([1, 80, 352])
  Audio shape: torch.Size([168960])
  Duration: 7.04s @ 24000Hz
  Saved to: ./results/sanity_check.wav



 Sanity check passed! Data pipeline is working correctly.


## Part 6: Training Loop

In this section, you will train the TextToSpeechLM model.

### Compute Requirements and Training Time

**Hardware Requirements:**
- **GPU Required**: This assignment requires a GPU with at least 16GB VRAM (e.g., NVIDIA T4, V100, A100, or similar)
- Training will not work on CPU-only machines due to memory and speed constraints

**Expected Training Time:**
- With default batch size (4), training typically takes **8-12 hours** on a T4 GPU for a full training run (3 epochs)
- Training time depends on:
  - GPU model
  - Batch size (larger batches = faster training but more memory)
  - Number of epochs
  - Data loading efficiency (num_workers)
- **Note**: If you have better GPUs like V100 or A100 with more VRAM, feel free to increase the batch size for faster training

**Checkpointing:**
- **It is strongly recommended to implement checkpointing** to save your model periodically during training
- This allows you to:
  - Resume training if interrupted (e.g., Colab disconnects, GPU timeout)
  - Save the best model based on validation loss
  - Avoid losing progress if training crashes
- Use the `save_checkpoint` helper function from `hw4_util.py` to save checkpoints after each epoch (or every N epochs)
- The helper function saves model state, optimizer state, scheduler state, epoch number, and losses for full resumability
- Consider saving checkpoints to Google Drive if using Colab to persist across sessions

### Training Components:
1. Setting up the optimizer and learning rate scheduler
2. Training for multiple epochs
3. Validating after each epoch
4. Saving the best model

**TODO:** Complete the training loop with proper loss computation and backpropagation.

**TODO:** Implement validation loop with metric tracking.

**TODO:** Add checkpointing to save the best model.


In [40]:
USE_AMP = torch.cuda.is_available()

# A100/H100 support bfloat16, which has fp32's exponent range -> no loss scaling needed.
# T4 only has fp16, which silently underflows small gradients without a GradScaler.
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and AMP_DTYPE == torch.float16))

GRAD_CLIP = 1.0
CHECKPOINT_EVERY_STEPS = 2000   # mid-epoch crash insurance

print(f"AMP: {USE_AMP} | dtype: {AMP_DTYPE} | GradScaler: {scaler.is_enabled()}")

def _batch_to_device(batch, device):
    """Move the four tensors the model needs onto the GPU."""
    return (
        batch['text_tokens'].to(device, non_blocking=True),
        batch['text_lengths'].to(device, non_blocking=True),
        batch['speech_tokens'].to(device, non_blocking=True),
        batch['speech_lengths'].to(device, non_blocking=True),
    )

def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Train for one epoch"""
    # TODO: Implement training loop
    model.train()
    total_loss, total_acc, num_batches = 0.0, 0.0, 0
    pbar = tqdm(dataloader, desc="Training", leave=False, mininterval=5.0)

    for step, batch in enumerate(pbar, start=1):

        if batch is None:
            continue

        # For each batch:
        # 1. Move data to device (GPU)
        text_tokens, text_lengths, speech_tokens, speech_lengths = _batch_to_device(batch, device)

        # 2. Forward pass to get loss and accuracy
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss, acc = model(text_tokens, text_lengths, speech_tokens, speech_lengths)

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        # 3. Backward pass (zero_grad → backward → clip_grad → step)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # 4. Update learning rate scheduler
        scheduler.step()

        # 5. Track metrics (accumulate loss and accuracy)
        total_loss += loss.item()
        total_acc += acc.item()
        num_batches += 1

        # 6. Update progress bar with current metrics
        if step % 50 == 0:
            pbar.set_postfix(
                loss=f"{total_loss / num_batches:.4f}",
                acc=f"{total_acc / num_batches:.3f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

        if CHECKPOINT_EVERY_STEPS and step % CHECKPOINT_EVERY_STEPS == 0:
            torch.save({'model_state_dict': model.state_dict(), 'step': step},
                       f'{Config.RESULTS_DIR}/last.pt')

    # Return: average_loss, average_accuracy
    return total_loss / max(num_batches, 1), total_acc / max(num_batches, 1)


@torch.no_grad()
def validate(model, dataloader, device):
    """Validate the model"""
    # TODO: Implement validation loop
    #
    # Similar to training but:
    # - Use model.eval() and torch.no_grad()
    # - No gradient computation or weight updates
    # - Only track loss and accuracy
    #
    # Return: average_loss, average_accuracy

    model.eval()
    total_loss, total_acc, num_batches = 0.0, 0.0, 0
    for batch in tqdm(dataloader, desc="Validating", leave=False, mininterval=5.0):
        if batch is None:
            continue

        text_tokens, text_lengths, speech_tokens, speech_lengths = _batch_to_device(batch, device)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss, acc = model(text_tokens, text_lengths, speech_tokens, speech_lengths)

        if not torch.isfinite(loss):
            continue

        total_loss += loss.item()
        total_acc += acc.item()
        num_batches += 1

    return total_loss / max(num_batches, 1), total_acc / max(num_batches, 1)

print("Training functions defined")


AMP: True | dtype: torch.bfloat16 | GradScaler: False
Training functions defined


In [41]:
from hw4_util import get_warmup_cosine_scheduler, save_checkpoint

import inspect, os, torch
print("save_checkpoint:", inspect.signature(save_checkpoint))
print("get_warmup_cosine_scheduler:", inspect.signature(get_warmup_cosine_scheduler))

# TODO: Setup training configuration
#
# Tasks:
# 1. Create AdamW optimizer (lr around 2e-4, weight_decay around 0.01)
# 2. Calculate warmup steps (e.g., 10% of total)
# 3. Create scheduler using get_warmup_cosine_scheduler
#
# The scheduler warms up learning rate then decays with cosine

num_epochs = 5

# 1.

learning_rate = 3e-4
weight_decay = 0.01
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
    betas=(0.9, 0.95),
)

# 2.
total_steps = len(train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)

# 3.
scheduler = get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)
print(f"AdamW(lr={learning_rate}, wd={weight_decay})")
print(f"Total steps: {total_steps:,} | Warmup steps: {warmup_steps:,}")

# Persist checkpoints to Drive so a dropped Colab session doesn't cost the whole run
DRIVE_CKPT_DIR = '/content/drive/MyDrive/hw4c_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(Config.RESULTS_DIR, exist_ok=True)


# Training info
steps_per_epoch = len(train_loader)
print(f"Steps per epoch: {steps_per_epoch}")

# Training loop
best_val_loss = float('inf')
train_losses = []
val_losses = []

print("\n" + "=" * 60)
print("Starting training...")
print("=" * 60 + "\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 60)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Validate
    val_loss, val_acc = validate(model, val_loader, device)
    val_losses.append(val_loss)

    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.2e}")

    # TODO: Save checkpoint when validation improves
    #
    # If val_loss < best_val_loss:
    # - Update best_val_loss
    # - Save checkpoint using save_checkpoint function
    # - Print confirmation message

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = f'{Config.RESULTS_DIR}/best_model.pt'
        try:
            save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, ckpt_path)
        except TypeError:
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, ckpt_path)

        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch,
                    'val_loss': val_loss}, f'{DRIVE_CKPT_DIR}/best_model_weights.pt')
        print(f"New best model saved (val_loss={val_loss:.4f}) -> {ckpt_path} + Drive")


print("\nTraining completed!")


save_checkpoint: (model, optimizer, scheduler, epoch, train_loss, val_loss, filepath)
get_warmup_cosine_scheduler: (optimizer, warmup_steps, total_steps)
AdamW(lr=0.0003, wd=0.01)
Total steps: 55,435 | Warmup steps: 5,543
Steps per epoch: 11087

Starting training...


Epoch 1/5
------------------------------------------------------------


Training:   0%|          | 0/11087 [00:00<?, ?it/s]

Validating:   0%|          | 0/312 [00:00<?, ?it/s]

Train Loss: 5.8450, Val Loss: 5.0190, LR: 2.91e-04
New best model saved (val_loss=5.0190) -> ./results/best_model.pt + Drive

Epoch 2/5
------------------------------------------------------------


Training:   0%|          | 0/11087 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/312 [00:00<?, ?it/s]

Train Loss: 4.3942, Val Loss: 4.4683, LR: 3.51e-05
New best model saved (val_loss=4.4683) -> ./results/best_model.pt + Drive

Epoch 5/5
------------------------------------------------------------


Training:   0%|          | 0/11087 [00:00<?, ?it/s]

Validating:   0%|          | 0/312 [00:00<?, ?it/s]

Train Loss: 4.3183, Val Loss: 4.4517, LR: 0.00e+00
New best model saved (val_loss=4.4517) -> ./results/best_model.pt + Drive

Training completed!


## Part 7: Inference and Evaluation

In this section, you will test your trained model by synthesizing speech.

### Tasks:
1. Load the best checkpoint
2. Generate speech from text
3. Support voice cloning with prompt audio

**TODO:** Load your trained checkpoint.

**TODO:** Generate audio samples and evaluate quality.


In [27]:
import os

LOCAL_CKPT = f'{Config.RESULTS_DIR}/best_model.pt'
DRIVE_CKPT = '/content/drive/MyDrive/hw4c_checkpoints/best_model_weights.pt'

ckpt_path = LOCAL_CKPT if os.path.exists(LOCAL_CKPT) else DRIVE_CKPT
print(f"Loading checkpoint: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt.get('model_state_dict', ckpt))
model.eval()
trained_model = model

print(f"Loaded epoch {ckpt.get('epoch')}, val_loss {ckpt.get('val_loss')}")

Loading checkpoint: /content/drive/MyDrive/hw4c_checkpoints/best_model_weights.pt
Loaded epoch 4, val_loss 4.451741464627095


In [15]:
print('speech_tokenizer' in globals(), 'flow_model' in globals(), 'text_tokenizer' in globals())


True False True


In [28]:
# Inference utilities
from hw4_util import load_trained_model, synthesize
import inspect, os

print("load_trained_model:", inspect.signature(load_trained_model))
print("synthesize:", inspect.signature(synthesize))

# Load pretrained components for inference
print("Loading pretrained components...")
flow_model = load_flow_model(model_dir, device)
vocoder = load_vocoder(model_dir, device)

# TODO: Load your trained model
#
# Check if checkpoint exists at Config.RESULTS_DIR/best.pt
# If yes: load using load_trained_model in hw4_util.py (needs path, vocab sizes, LM, device)
# If no: use current model state

LOCAL_CKPT = f'{Config.RESULTS_DIR}/best_model.pt'
DRIVE_CKPT = '/content/drive/MyDrive/hw4c_checkpoints/best_model_weights.pt'
ckpt_path = next((p for p in (LOCAL_CKPT, DRIVE_CKPT) if os.path.exists(p)), None)
if ckpt_path is None:
    print("No checkpoint found - using current in-memory model state")
    trained_model = model
else:
    print(f"Loading checkpoint: {ckpt_path}")
    try:
        trained_model = load_trained_model(
            ckpt_path, text_tokenizer.vocab_size, speech_vocab_size, TextToSpeechLM, device
        )
    except Exception as e:
        print(f"  load_trained_model failed ({e}) - loading state_dict directly")
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt.get('model_state_dict', ckpt))
        trained_model = model
    trained_model.to(device).eval()
    print("Checkpoint loaded")

# Test texts
test_texts = [
    "Hello, this is a test of the trained model.",
    "Text to speech synthesis with language models.",
    "CosyVoice two is a powerful speech synthesis system."
]

print("\nTesting TTS synthesis...")

# TODO: Generate audio for test texts
#
# For each text:
# - Use synthesize function to generate audio
# - Save to Config.RESULTS_DIR/synthesized_{i}.wav
# - Display audio using IPython.display.Audio
#
# synthesize returns (audio_tensor, sample_rate)

for i, text in enumerate(test_texts):
    print(f"\n[{i+1}/{len(test_texts)}] '{text}'")
    output_path = f'{Config.RESULTS_DIR}/synthesized_{i}.wav'

    audio, sr = synthesize(
        text=text,
        model=trained_model,
        text_tokenizer=text_tokenizer,
        speech_tokenizer=speech_tokenizer,
        flow_model=flow_model,
        vocoder=vocoder,
        device=device,
        output_path=output_path,
    )

    print(f"  Duration: {audio.shape[-1] / sr:.2f}s @ {sr}Hz -> {output_path}")
    display(Audio(audio.cpu().numpy(), rate=sr))

print("\nInference completed!")


load_trained_model: (checkpoint_path, text_vocab_size, speech_vocab_size, TextToSpeechLM, device='cuda')
synthesize: (text, model, text_tokenizer, speech_tokenizer, flow_model, vocoder, device='cuda', prompt_audio=None, prompt_text='', output_path=None)
Loading pretrained components...
  - Checkpoint: flow.pt
  - Device: cuda
  - Checkpoint: hift.pt
  - Device: cuda
Loading checkpoint: /content/drive/MyDrive/hw4c_checkpoints/best_model_weights.pt


/tmp/ipykernel_1240/2415452517.py:54: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Loaded model from checkpoint (epoch 4)
  load_trained_model failed ('train_loss') - loading state_dict directly
Checkpoint loaded

Testing TTS synthesis...

[1/3] 'Hello, this is a test of the trained model.'

Synthesizing: 'Hello, this is a test of the trained model.'
Generated: 2.64s @ 24000Hz
Saved to: ./results/synthesized_0.wav
  Duration: 2.64s @ 24000Hz -> ./results/synthesized_0.wav



[2/3] 'Text to speech synthesis with language models.'

Synthesizing: 'Text to speech synthesis with language models.'
Generated: 2.64s @ 24000Hz
Saved to: ./results/synthesized_1.wav
  Duration: 2.64s @ 24000Hz -> ./results/synthesized_1.wav



[3/3] 'CosyVoice two is a powerful speech synthesis system.'

Synthesizing: 'CosyVoice two is a powerful speech synthesis system.'
Generated: 3.76s @ 24000Hz
Saved to: ./results/synthesized_2.wav
  Duration: 3.76s @ 24000Hz -> ./results/synthesized_2.wav



Inference completed!


## Part 8: Voice Cloning and Final Evaluation

This is the final evaluation section where you will record your voice directly in the notebook and use it for voice cloning. You will generate 200 utterances with your cloned voice and evaluate the quality using Whisper ASR.

**TODO:** Record your voice prompt.

**TODO:** Run the voice cloning evaluation.

**TODO:** Submit results to Gradescope (included in evaluation above).


### Step 1: Record Your Voice

In [30]:
from hw4_util import record_or_load_voice

# Record or load existing voice
prompt_audio, prompt_text, recorded_file_path = record_or_load_voice(
    prompt_text="The quick brown fox jumps over the lazy dog.",
    results_dir=Config.RESULTS_DIR
)



EXISTING RECORDING FOUND
Loading from: ./results/voice_prompt_16k.wav
Duration: 9.06s
Sample rate: 16000Hz

Playback of existing recording:



To record a new voice:
   1. Delete the file: ./results/voice_prompt_16k.wav
   2. Re-run this cell
   Or call: record_or_load_voice(..., force_new=True)


### Step 2: Test Voice Cloning with One Utterance

In [32]:
print("=" * 60)
print("TESTING VOICE CLONING")
print("=" * 60)

# Test with a single utterance first
# You may play around with the test text to see how the model performs
test_text = "This is my cloned voice speaking."
print(f"\n Test text: '{test_text}'")
print("Generating with your voice...")

# Generate with voice cloning
audio_test, sr = synthesize(
    text=test_text,
    model=trained_model,
    text_tokenizer=text_tokenizer,
    speech_tokenizer=speech_tokenizer,
    flow_model=flow_model,
    vocoder=vocoder,
    device=device,
    prompt_audio=prompt_audio,
    prompt_text=prompt_text,
    output_path=f'{Config.RESULTS_DIR}/voice_clone_test.wav'
)

print("\n Generated audio with your cloned voice:")
display(Audio(audio_test.cpu().numpy(), rate=sr))

print("\n Voice cloning test completed!")
print("If this sounds like your voice, proceed to the next cell.")
print("If not, try recording again with clearer pronunciation.")


TESTING VOICE CLONING

 Test text: 'This is my cloned voice speaking.'
Generating with your voice...

Synthesizing: 'This is my cloned voice speaking.'
Voice cloning mode with prompt: 'The quick brown fox jumps over the lazy dog.'
Using voice cloning with prompt audio
Generated: 2.48s @ 24000Hz
Saved to: ./results/voice_clone_test.wav

 Generated audio with your cloned voice:



 Voice cloning test completed!
If this sounds like your voice, proceed to the next cell.
If not, try recording again with clearer pronunciation.


### Step 3: Large-Scale Evaluation with ASR

**Automated evaluation system:**
- All students evaluate the same 200 texts
- Text order shuffled by Student ID (prevents cheating)
- Submit to Gradescope for WER scoring

**Process:**
1. Enter your Student ID → unique shuffle
2. Generate 200 utterances (~10-15 min)
3. Transcribe with Whisper (~5-10 min)
4. Submit `submission_[ID].txt` to Gradescope


In [33]:
import torch

# Load pre-generated fixed test set
# (All students use the same 200 texts)
fixed_test_path = f'{Config.DATA_CACHE_DIR}/fixed_test_set.pt'
test_data = torch.load(fixed_test_path)

FIXED_TEST_TEXTS = test_data['texts']
FIXED_TEST_INDICES = test_data['indices']

print(f"Loaded {len(FIXED_TEST_TEXTS)} test texts from {fixed_test_path}")
print(f"Seed: {test_data['seed']}")
print(f"Distribution: {test_data['n_short']} short + {test_data['n_medium']} medium + {test_data['n_long']} long")


Loaded 200 test texts from ./libritts_token_cache/fixed_test_set.pt
Seed: 42
Distribution: 60 short + 100 medium + 40 long


### Step 4: Voice Cloning Evaluation (200 Utterances)

**Three-step process:**

1. **Load models:** Text tokenizer, speech tokenizer, flow model, vocoder
2. **Enter Student ID:** Generates unique text shuffling seed
3. **Run evaluation:**
   - Generate 200 utterances with your voice (~10-15 min)
   - Transcribe with Whisper ASR (~5-10 min)
   - Create Gradescope submission file

**Note:** WER score visible only on Gradescope after submission.


In [35]:
# Voice Cloning Evaluation
# Implementation details are in hw4_util.py

from hw4_util import load_pretrained_models_for_inference, run_voice_cloning_evaluation

# Step 1: Load pretrained models
print("Step 1: Loading pretrained models...\n")
models = load_pretrained_models_for_inference(
    pretrained_dir=Config.PRETRAINED_DIR,
    device=device
)

# Extract models
text_tokenizer = models['text_tokenizer']
speech_tokenizer = models['speech_tokenizer']
flow_model = models['flow_model']
vocoder = models['vocoder']

# Step 2: Get student ID
print("\nStep 2: Student identification\n")
STUDENT_ID = input("Enter your Student ID: ").strip()

print(f" Student ID: STUDENT_ID")

# Step 3: Run evaluation
print("\nStep 3: Running voice cloning evaluation...\n")
submission_path = run_voice_cloning_evaluation(
    student_id=STUDENT_ID,
    trained_model=trained_model,
    text_tokenizer=text_tokenizer,
    speech_tokenizer=speech_tokenizer,
    flow_model=flow_model,
    vocoder=vocoder,
    prompt_audio=prompt_audio,
    prompt_text=prompt_text,
    fixed_test_set_path=f'{Config.DATA_CACHE_DIR}/fixed_test_set.pt',
    results_dir=Config.RESULTS_DIR,
    pretrained_dir=Config.PRETRAINED_DIR,
    device=device
)

print(f"\n Done! Upload {submission_path} to Gradescope")


Step 1: Loading pretrained models...

LOADING PRETRAINED MODELS FOR INFERENCE

Checking pretrained models...


2026-07-26 17:04:26,206 | INFO    | modelscope_hub.download | Downloading 21 files from iic/CosyVoice2-0.5B@master


Downloading:   0%|          | 0/21 [00:00<?, ?file/s]

Downloaded to: pretrained_models/models/iic--CosyVoice2-0.5B/snapshots/master

Loading text tokenizer (Qwen2)...
Text tokenizer loaded (Qwen2 BPE)
  Vocab size: 151643
   Vocab size: 151643

Loading speech tokenizer (FSQ)...
   Vocab size: 6561

Loading flow matching model...
  - Checkpoint: flow.pt
  - Device: cuda
   Flow model loaded

Loading vocoder (HiFi-GAN)...
  - Checkpoint: hift.pt
  - Device: cuda
   Vocoder loaded

ALL MODELS LOADED

Step 2: Student identification

Enter your Student ID: 3038468123
 Student ID: STUDENT_ID

Step 3: Running voice cloning evaluation...


VOICE CLONING EVALUATION

Student ID: 3038468123

Loading fixed test set...
Loaded 200 test texts
Shuffle seed: 1882290033

Loading Whisper ASR model...
   Cache directory: ./pretrained_models/whisper


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

Using voice cloning with prompt audio


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using voice cloning with prompt audio


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Successfully processed: 200/200

Generating submission file...

EVALUATION COMPLETE

Statistics:
   - Total utterances: 200
   - Successfully processed: 200
   - Failed: 0

Submission file: ./results/submission_3038468123.txt
   Size: 9.9 KB

Upload submission_3038468123.txt to Gradescope
   WER score will be visible after submission

 Done! Upload ./results/submission_3038468123.txt to Gradescope


In [36]:
from google.colab import files
files.download('./results/submission_3038468123.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 9: Final Submission Checklist

Before submitting to Gradescope, ensure you have completed the following:

### Required Files:

1. **`submission_[YOUR_ID].txt`** - Auto-generated evaluation file from Part 8
   - Generated automatically when you complete Part 8
   - Contains your Student ID, seed, and 200 transcriptions
   - Do NOT modify this file

2. **`hw4-c.pdf`** - PDF export of this notebook

### Submission Instructions:
1. Complete Part 8 to generate `submission_[YOUR_ID].txt`
2. Export this notebook to PDF
3. Upload both files to Gradescope

### Notes:
* Your WER score will be calculated automatically upon submission to GradScope
* The points distribution based on WER on the test set:

   - < 70% : 15 points

   - < 50% : 20 points

   - < 45% : 25 points

   - < 40% : 30 points

   - < 35% : 35 points

   - < 30% : 40 points (Full points)